<a href="https://colab.research.google.com/github/Aakash-cyber007/Private-LLM-text-generation-with-high-utility/blob/main/Toy_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gumbel_r, expon
from plotly.subplots import make_subplots

In [7]:
# ==========================================
# 1. Define DP Parameters
# ==========================================
epsilon = 2.0
delta_q = 1.0
scale = (2 * delta_q) / epsilon  # This equals 1.0 for both distributions

# Generate x-axis values
x = np.linspace(-3, 8, 1000)

# ==========================================
# 2. Calculate Probability Density Functions (PDF)
# ==========================================
# Gumbel Noise (Softmax SOTA) - Centered at 0
pdf_gumbel = gumbel_r.pdf(x, loc=0, scale=scale)

# Exponential Noise (RNM) - Starts at 0
# Scipy's expon is 0 for x < 0 automatically
pdf_expo = expon.pdf(x, loc=0, scale=scale)

# ==========================================
# 3. Create the Beautiful Plotly Figure
# ==========================================
fig = go.Figure()

# Add Gumbel Trace
fig.add_trace(go.Scatter(
    x=x, y=pdf_gumbel,
    mode='lines',
    name='Gumbel Noise (Softmax SOTA)',
    line=dict(color='#1f77b4', width=3),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.2)'
))

# Add Exponential Trace
fig.add_trace(go.Scatter(
    x=x, y=pdf_expo,
    mode='lines',
    name='Exponential Noise (RNM)',
    line=dict(color='#ff7f0e', width=3),
    fill='tozeroy',
    fillcolor='rgba(255, 127, 14, 0.2)'
))

# Format the Layout
fig.update_layout(
    title=dict(
        text=rf'Noise Distribution Comparison at Strict Privacy ($\\epsilon$ = {epsilon})',
        font=dict(size=20, family="Arial, bold")
    ),
    xaxis_title='Noise Value Added to Logit',
    yaxis_title='Probability Density',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        yanchor="top", y=0.99,
        xanchor="right", x=0.99,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black", borderwidth=1
    ),
    font=dict(size=14)
)

fig.show()

In [12]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. Setup Data & Simulation (Same as before)
# ==========================================
vocab_size = 100
logits = np.array([15.0 - 2.5 * np.log(i + 1) for i in range(vocab_size)])

epsilon = 2.0
delta_q = 1.0
num_trials = 10000

def gumbel_max(logits, eps, delta_q):
    scale = (2 * delta_q) / eps
    noise = np.random.gumbel(loc=0.0, scale=scale, size=len(logits))
    return np.argmax(logits + noise)

def rnm_exponential(logits, eps, delta_q):
    scale = (2 * delta_q) / eps
    noise = np.random.exponential(scale=scale, size=len(logits))
    return np.argmax(logits + noise)

def permute_and_flip(logits, eps, delta_q):
    q_max = np.max(logits)
    indices = np.random.permutation(len(logits))
    for idx in indices:
        p_i = np.exp((eps / (2 * delta_q)) * (logits[idx] - q_max))
        if np.random.rand() <= p_i:
            return idx
    return np.argmax(logits)

print("Running simulation...")
counts_gumbel = np.zeros(vocab_size)
counts_rnm = np.zeros(vocab_size)
counts_pf = np.zeros(vocab_size)

for _ in range(num_trials):
    counts_gumbel[gumbel_max(logits, epsilon, delta_q)] += 1
    counts_rnm[rnm_exponential(logits, epsilon, delta_q)] += 1
    counts_pf[permute_and_flip(logits, epsilon, delta_q)] += 1

prob_gumbel = counts_gumbel / num_trials
prob_rnm = counts_rnm / num_trials
prob_pf = counts_pf / num_trials
x_axis = np.arange(vocab_size)

# ==========================================
# 2. Build the Plotly Dashboard
# ==========================================
fig = go.Figure()

# TRACE 1: Softmax / Gumbel (Bright Red)
fig.add_trace(go.Scatter(
    x=x_axis, y=prob_gumbel,
    mode='lines',
    name='Softmax (Gumbel SOTA)',
    line=dict(color='#E63946', width=3),
    fill='tozeroy', fillcolor='rgba(230, 57, 70, 0.1)'
))

# TRACE 2: RNM (Deep Blue, thick and dotted)
fig.add_trace(go.Scatter(
    x=x_axis, y=prob_rnm,
    mode='lines',
    name='RNM + Exp Noise',
    line=dict(color='#457B9D', width=7, dash='dot'),
    opacity=0.6
))

# TRACE 3: Permute-and-Flip (Neon Green, solid thin)
fig.add_trace(go.Scatter(
    x=x_axis, y=prob_pf,
    mode='lines',
    name='Permute-and-Flip',
    line=dict(color='#2A9D8F', width=2)
))

# ==========================================
# 3. Add Horizontal Toggle Buttons
# ==========================================
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",          # Changed from dropdown to buttons
            direction="right",       # Lays them out horizontally
            active=0,                # Which button is pressed by default
            x=0.5,                   # Center the buttons
            xanchor="center",
            y=1.12,                  # Place them just above the chart
            yanchor="bottom",
            showactive=True,
            bgcolor="white",
            bordercolor="gray",
            buttons=list([
                dict(label="Show All",
                     method="update",
                     args=[{"visible": [True, True, True]},
                           {"title": "All Mechanisms: Distribution Comparison"}]),
                dict(label="Compare: PF vs Softmax",
                     method="update",
                     args=[{"visible": [True, False, True]},
                           {"title": "Utility Proof: Permute-and-Flip removes the Softmax Heavy Tail"}]),
                dict(label="Compare: PF vs RNM",
                     method="update",
                     args=[{"visible": [False, True, True]},
                           {"title": "Equivalence Proof: PF perfectly tracks RNM Exponential Noise"}])
            ])
        )
    ]
)

# Format axes and zoom in on the action
fig.update_layout(
    title_text="Interactive DP Decoding Distributions",
    xaxis_title="Token Rank (0 = Best Token, 99 = Worst)",
    yaxis_title="Probability of Selection",
    template='plotly_white',
    hovermode='x unified',
    xaxis=dict(range=[0, 30]), # ZOOM IN on the top 30 tokens!
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)

fig.show()

Running simulation...
